## Notebook for testing our pipeline for the Generative Recommender

In [1]:
# Imports here
from DataHandler import get_article_feature_string_list
from embedder import Embedder
from rqvae import RQVAE
from rqvae_train import rqvae_loss, train_rqvae_sanity_check, train_rqvae_full, load_trained_rqvae
import torch
import numpy as np
import pickle
import os
import joblib
import torch, numpy as np, random
import pandas as pd
from transformer import get_all_unique_sids, prepare_dataset, train_model, recommended_next_sid, is_model_trained
from transformers import BartTokenizer, BartForConditionalGeneration
from collaborative_filtering import compute_item_user_matrix, recommend_for_user


# This is ONLY for our tests! Do not use for our full model
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

/opt/anaconda3/envs/devenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### First, get the feature strings from DataHandler

In [2]:
feature_strings = get_article_feature_string_list()
# feature_strings is a list of lists, use list comprehension to make it a list of strings (first 1000 items)
feature_strings = [item[0] for item in feature_strings]
item_ids = [feature_string.split()[0] for feature_string in feature_strings]

print(f'Number of items: {len(feature_strings)}')
print(feature_strings[0])


Number of items: 105542
108775015 Solid Dark Black Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic


### Next, we use the embedder to get the embeddings that will serve as input for RQ-VAE

In [3]:
if os.path.exists('SBERT_embeddings_fulldata.npy'):
    embeddings = np.load('SBERT_embeddings_fulldata.npy')
    print(f'The embeddings have the shape: {embeddings.shape}')
else:
    embedder = Embedder("sbert")
    embeddings = embedder.encode(feature_strings) # embeddings has shape (n, d), where d = 384 for SBERT
    print(f'The embeddings have the shape: {embeddings.shape}')
    # print(embeddings[0])
    np.save('SBERT_embeddings_fulldata.npy', embeddings)

The embeddings have the shape: (105542, 384)


### Now, we can use the embeddings with RQ-VAE. The code loads hashmaps between item IDs and semantic IDs if they exist. If not, it uses the model to generate semantic IDs and creates (and saves) the hashmaps

In [4]:
input_dim = embeddings.shape[1]
latent_dim = 32
hidden_dim = 256
codebook_size = 512
num_quantizers = 4

TRAINING_MODE = 'None' # for loading

set_seed(42)
rqvae = RQVAE(input_dim=input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim, codebook_size=512, num_quantizers=num_quantizers)

# We need to train the model here, before retrieving semantic IDs! Omitting for now... 
# trained_model = ...

print('Generating semantic IDs')

if isinstance(embeddings, np.ndarray):
    embeddings = torch.from_numpy(embeddings).float()

if TRAINING_MODE == 'sanity_check':
    print("Running sanity check for RQ-VAE")
    rqvae = train_rqvae_sanity_check(rqvae, embeddings)

elif TRAINING_MODE == 'full_training':
    print("Training RQ-VAE")
    rqvae = train_rqvae_full(rqvae, embeddings, save_path='./models/rqvae')

else:
    rqvae = load_trained_rqvae(rqvae, 'quantizers/rqvae_training_results_20251101/models/rqvae_20251101_best.pth')

semantic_ids = rqvae.encode_to_semantic_ids(embeddings)
semantic_ids = semantic_ids.detach().numpy() # List of semantic IDs

print(f'semantic_ids has shape: {semantic_ids.shape}')
print('printing the first semantic ID (trained RQ-VAE)')
print(semantic_ids[0])

item_2_semantic = {}
semantic_2_item = {}

if os.path.exists('quantizers/rqvae_training_results_20251101/item_2_semantic.pkl') and os.path.exists('quantizers/rqvae_training_results_20251101/semantic_2_item.pkl'):
    print('Loading existing hashmaps')

    with open('quantizers/rqvae_training_results_20251101/item_2_semantic.pkl', 'rb') as f:
        item_2_semantic = pickle.load(f)
    with open('quantizers/rqvae_training_results_20251101/semantic_2_item.pkl', 'rb') as f:
        semantic_2_item = pickle.load(f)
    print('Loaded the hashmaps')

else:
    for item_id, semantic_id in zip(item_ids, semantic_ids):
        semantic_tuple = tuple(semantic_id.astype(int))
        item_2_semantic[item_id] = semantic_tuple
        semantic_2_item[semantic_tuple] = item_id

    print(f'Done creating hashmaps. Saving...')
    with open('item_2_semantic.pkl', 'wb') as f:
        pickle.dump(item_2_semantic, f)

    with open('semantic_2_item.pkl', 'wb') as f:
        pickle.dump(semantic_2_item, f)
    print(f'Saved hashmaps to files.')

Generating semantic IDs
semantic_ids has shape: (105542, 4)
printing the first semantic ID (trained RQ-VAE)
[201  91 394  18]
Loading existing hashmaps
Loaded the hashmaps


### Converting transactions data into: 
#### i.e.: user_id: item_id1, ..., item_idn --> user_id: item_SID1, ..., item_SIDn

In [5]:
transactions_train = pd.read_pickle('customer_transactions_TRAIN60P.pkl')
transactions_val = pd.read_pickle('customer_transactions_VAL20P.pkl')

customer_transactions_train = {}
customer_transactions_val = {}
if os.path.exists('customer_transactions_train.pkl') and os.path.exists('customer_transactions_val.pkl'):
    print('Loading existing customer_transactions_train and customer_transactions_val')

    with open('customer_transactions_train.pkl', 'rb') as f:
        customer_transactions_train = pickle.load(f)
    with open('customer_transactions_val.pkl', 'rb') as f:
        customer_transactions_val = pickle.load(f)

    print('Loaded existing customer_transactions_train and customer_transactions_val')
else:

    for customer_id, group in transactions_train.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_train[customer_id] = sid_list
    
    for customer_id, group in transactions_val.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_val[customer_id] = sid_list
    
    print("customer_transactions_train and customer_transactions_val have been created, saving...")
    with open("customer_transactions_train.pkl", "wb") as f:
        pickle.dump(customer_transactions_train, f)
    with open('customer_transactions_val.pkl', 'wb') as f:
        pickle.dump(customer_transactions_val, f)
    print("Saving done!")

first_customer = list(customer_transactions_val.keys())[0]
print(f"{first_customer}: {customer_transactions_val[first_customer]}")

Loading existing customer_transactions_train and customer_transactions_val
Loaded existing customer_transactions_train and customer_transactions_val
00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a: [(np.int64(126), np.int64(6), np.int64(235), np.int64(74)), (np.int64(289), np.int64(352), np.int64(327), np.int64(7)), (np.int64(101), np.int64(176), np.int64(218), np.int64(256)), (np.int64(169), np.int64(81), np.int64(326), np.int64(31)), (np.int64(275), np.int64(458), np.int64(343), np.int64(237)), (np.int64(275), np.int64(458), np.int64(343), np.int64(237)), (np.int64(25), np.int64(202), np.int64(311), np.int64(19)), (np.int64(441), np.int64(253), np.int64(43), np.int64(351)), (np.int64(353), np.int64(136), np.int64(82), np.int64(317)), (np.int64(185), np.int64(269), np.int64(375), np.int64(258)), (np.int64(247), np.int64(41), np.int64(343), np.int64(39)), (np.int64(175), np.int64(195), np.int64(85), np.int64(114)), (np.int64(413), np.int64(299), np.int64(2), np.int64(6

### Taking a subset of transaction data (customer_transations_train, customer_transactions_val) for training of the transfomer

In [6]:
train_subset_keys = list(customer_transactions_train.keys())[:200]
customer_transactions_train_subset = {k: customer_transactions_train[k] for k in train_subset_keys}
print("First sample of train data: ", customer_transactions_train_subset[list(customer_transactions_train_subset.keys())[0]])

val_subset_keys = list(customer_transactions_val.keys())[:50]
customer_transactions_val_subset = {k: customer_transactions_val[k] for k in val_subset_keys}
print("First sample of val data: ", customer_transactions_val_subset[list(customer_transactions_val_subset.keys())[0]])

First sample of train data:  [(np.int64(139), np.int64(283), np.int64(20), np.int64(354)), (np.int64(462), np.int64(79), np.int64(375), np.int64(252)), (np.int64(288), np.int64(108), np.int64(30), np.int64(348)), (np.int64(218), np.int64(202), np.int64(120), np.int64(403)), (np.int64(502), np.int64(423), np.int64(113), np.int64(70)), (np.int64(502), np.int64(423), np.int64(113), np.int64(70)), (np.int64(321), np.int64(423), np.int64(330), np.int64(94)), (np.int64(381), np.int64(122), np.int64(356), np.int64(231)), (np.int64(260), np.int64(261), np.int64(303), np.int64(53)), (np.int64(502), np.int64(219), np.int64(312), np.int64(310)), (np.int64(502), np.int64(219), np.int64(312), np.int64(310)), (np.int64(303), np.int64(219), np.int64(120), np.int64(30)), (np.int64(318), np.int64(249), np.int64(505), np.int64(343)), (np.int64(289), np.int64(41), np.int64(357), np.int64(258)), (np.int64(482), np.int64(272), np.int64(112), np.int64(24)), (np.int64(63), np.int64(436), np.int64(492), np.in

### Training the transformer

In [7]:
window_size=3

if is_model_trained(): 
    print('The model is loaded...')
    model = BartForConditionalGeneration.from_pretrained('./bart-recommender/final_model')
    tokenizer = BartTokenizer.from_pretrained('./bart-recommender/final_model')
else: 
    print('There is no pretrained model, the model will be trained ...')

    unique_sids_train = get_all_unique_sids(customer_transactions_train_subset)
    unique_sids_val = get_all_unique_sids(customer_transactions_val_subset)
    all_unique_sids = set(unique_sids_train)
    for sid in unique_sids_val:
        all_unique_sids.add(sid)
    
    tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')
    tokenizer.add_tokens(list(all_unique_sids))

    model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
    # resize model embeddings after adding token to tokenizer
    model.resize_token_embeddings(len(tokenizer))

    train_dataset = prepare_dataset(customer_transactions_train_subset, window_size, tokenizer)
    val_dataset = prepare_dataset(customer_transactions_val_subset, window_size, tokenizer)
    train_model(train_dataset, model, val_dataset)

    model.save_pretrained('./bart-recommender/final_model')
    tokenizer.save_pretrained('./bart-recommender/final_model')

The model is loaded...


### Retrieving articles info for mapping

In [8]:
articles = pd.read_pickle('articles.pkl')
article_id = 108775044
row = articles.loc[articles['article_id'] == article_id]
product_name = row['prod_name']
product_name2 = row['prod_name'].iloc[0]
print(product_name)
print(product_name2)

1    Strap top
Name: prod_name, dtype: object
Strap top


### Inference: 

### GR: recommend SIDs for given customer transaction sequence

In [9]:
last_10_customers = list(customer_transactions_val.keys())[-10:]
test_customers_sequences = {k: customer_transactions_val[k] for k in last_10_customers}

for customer_id, seqs in test_customers_sequences.items():
    test_seq = test_customers_sequences[customer_id][-window_size:]
    test_seq = [' '.join(tuple(str(x) for x in sid)) for sid in test_seq]
    print(f"Customer: {customer_id}")
    print(f"test SIDs sequence: {test_seq}")
    recommended_sids = recommended_next_sid(test_seq, model, tokenizer, window_size, top_k=12)
    filtered = [ # filter out empty sids
        (sid, semantic_2_item[tuple(int(x) for x in sid.split())])
        for sid in recommended_sids
        if sid.strip() != ''
    ]

    print("Recommended SIDs, generated by transformer: ")
    for sid, id in filtered:
        row = articles.loc[articles['article_id'].astype(str) == str(id)]
        prod_name = row['prod_name'].iloc[0]
        print(f'Rec. SID: {sid} --> article_id: {id} --> product_name: {prod_name}')

Customer: fffd70b14382482cf3d10def86df367b2b6e0af711dede538a6a036dcc04da0b
test SIDs sequence: ['368 219 505 35', '481 204 380 510', '16 155 223 187']
Recommended SIDs, generated by transformer: 
Rec. SID: 394 425 308 56 --> article_id: 699080001 --> product_name: Lazer Razer Padded Wire
Rec. SID: 314 18 335 20 --> article_id: 741356009 --> product_name: Pamela Shorts HW
Rec. SID: 469 279 90 133 --> article_id: 372860002 --> product_name: 7p Basic Shaftless
Rec. SID: 446 269 349 81 --> article_id: 854043005 --> product_name: Trailmix Seamless bralette
Rec. SID: 446 74 280 237 --> article_id: 754238024 --> product_name: Karin padded sport bra
Rec. SID: 30 308 60 252 --> article_id: 448509028 --> product_name: Perrie Slim Mom Denim TRS
Rec. SID: 159 210 214 39 --> article_id: 841383002 --> product_name: Vanessa 2-pack
Rec. SID: 63 418 497 404 --> article_id: 717490075 --> product_name: Cat Tee.
Rec. SID: 368 168 120 193 --> article_id: 677930086 --> product_name: Queen Sweater
Rec. SID: 

### 2. Collaborative Filtering:

In [10]:
if os.path.exists('CF_cashe.joblib'):
    unique_items, user2idx, interaction_matrix, item_similarity = joblib.load('CF_cashe.joblib')
else:
    unique_items, user2idx, interaction_matrix, item_similarity = compute_item_user_matrix()
    joblib.dump((unique_items,user2idx, interaction_matrix, item_similarity), "CF_cashe.joblib")

for customer_id in test_customers_sequences:
    recommended_items = recommend_for_user(customer_id, unique_items, user2idx, interaction_matrix, item_similarity, top_k=12)
    print(f"Customer: {customer_id}")
    for article_id in recommended_items:
        row = articles.loc[articles['article_id'].astype(str) == str(article_id)]
        prod_name = row['prod_name'].iloc[0]
        print(f"article_id: {id} --> product_name: {prod_name}")

Customer: fffd70b14382482cf3d10def86df367b2b6e0af711dede538a6a036dcc04da0b
article_id: 832330003 --> product_name: Simple as that Cheeky Tanga
article_id: 832330003 --> product_name: Timeless Cheeky Brief
article_id: 832330003 --> product_name: Timeless Midrise Brief
article_id: 832330003 --> product_name: Tilly (1)
article_id: 832330003 --> product_name: Jade HW Skinny Denim TRS
article_id: 832330003 --> product_name: Brit Baby Tee
article_id: 832330003 --> product_name: Perrie Slim Mom Denim TRS
article_id: 832330003 --> product_name: Jade HW Skinny Denim TRS
article_id: 832330003 --> product_name: Tilda tank
article_id: 832330003 --> product_name: Lazer Razer Brief
article_id: 832330003 --> product_name: Chestnut strap top
article_id: 832330003 --> product_name: Pamela Shorts HW
Customer: fffd82f00fc748fae98fa569c6863a4c5b2242e0c8162871a91bcbf10b4b95be
article_id: 832330003 --> product_name: JOGGER CARGO ANKLE
article_id: 832330003 --> product_name: WAFFLE LS CTN SLIM FIT
article_id